# Chirp Generation

In [1]:
import numpy as np
from scipy.io import wavfile
from scipy.signal import windows

# 1. Core Acoustic Parameters
fs = 44100          # 44.1 kHz sampling rate (Standard CD quality)
f_start = 18000     # Start frequency of LFM sweep (Hz)
f_end = 22000       # End frequency of LFM sweep (Hz)
chirp_duration = 0.010  # 10 milliseconds
ping_interval = 0.100   # 100 milliseconds total cycle (10 Hz ping rate)

# 2. Generate a Single LFM Chirp Vector
t_chirp = np.linspace(0, chirp_duration, int(fs * chirp_duration), endpoint=False)
# Linear Frequency Modulation formula
chirp = np.sin(2 * np.pi * (f_start + (f_end - f_start) * t_chirp / (2 * chirp_duration)) * t_chirp)

# 3. Apply a Tukey Window (Tapered Cosine) to smooth the edges and eliminate "clicks"
# alpha=0.1 means the first and last 5% of the signal are smoothly ramped up/down
window = windows.tukey(len(chirp), alpha=0.1)
tapered_chirp = chirp * window

# 4. Construct the Loop Sequence (e.g., 60 seconds of walking data)
total_duration = 30  # seconds
num_pings = int(total_duration / ping_interval)
samples_per_interval = int(fs * ping_interval)
samples_per_chirp = len(tapered_chirp)

# Initialize a mono array of silence
mono_sequence = np.zeros(fs * total_duration)

# Plant the chirps rhythmically into the timeline
for i in range(num_pings):
    start_idx = i * samples_per_interval
    end_idx = start_idx + samples_per_chirp
    # Ensure we don't overflow the array boundaries
    if end_idx < len(mono_sequence):
        mono_sequence[start_idx:end_idx] = tapered_chirp

# 5. Create a Master Stereo Track with Software Panning
# We initialize a 2D array of zeros [samples, channels]
stereo_master = np.zeros((len(mono_sequence), 2))

# FORCE THE CHIRP ENTIRELY TO THE LEFT CHANNEL
# This isolates a single point-source speaker on your device!
stereo_master[:, 0] = mono_sequence  # Left channel gets the pings
stereo_master[:, 1] = 0.0            # Right channel is absolute silence

# 6. Normalize to prevent clipping and convert to 16-bit PCM WAV
stereo_master = stereo_master / np.max(np.abs(stereo_master))
audio_int16 = (stereo_master * 32767).astype(np.int16)

# 7. Export the asset
wavfile.write('Chirp/master_chirp_sequence.wav', fs, audio_int16)
print("Master chirp track exported successfully to Chirp/master_chirp_sequence.wav")

Master chirp track exported successfully to Chirp/master_chirp_sequence.wav
